# 外部数据与运行时校验

学习目标：把不可信输入转成经过运行时检查的数据，并为失败和 DOM 操作建立明确边界。

前置知识：unknown、联合类型、类型收窄、Promise 与异常、基础 DOM 查询和事件。

适用版本与条件：TypeScript 7.0.2、Node.js 24.11.0；使用 ES 模块，开启 strict。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/22-runtime-validation/。

1. [validation.ts](scripts/22-runtime-validation/validation.ts)：结构守卫、业务约束、JSON 解析与结果类型。
2. [main.ts](scripts/22-runtime-validation/main.ts)：成功、失败与断言函数的调用。
3. [validation.test.ts](scripts/22-runtime-validation/validation.test.ts)：无效值及业务边界测试。
4. [browser.ts](scripts/22-runtime-validation/browser.ts)：独立浏览器类型配置下的事件处理。
5. [index.html](scripts/22-runtime-validation/index.html)：数量输入页面。
6. [tsconfig.json](scripts/22-runtime-validation/tsconfig.json)：Node 项目；另有 tsconfig.browser.json 与 tsconfig.errors.json。

## 1 把外部输入留在 unknown

本章专门学习运行时校验，故意提供不合法输入来比较失败阶段；这些检查属于当前教学机制，不是其他章节默认需要的防护模板。

JSON.parse 先检查 JSON 文本语法，合法 JSON 仍可能是 null、数组或不含业务字段的对象。标准库声明不能保证解析结果就是订单，因此本例马上赋给 unknown，不用 as Order 假定其结构。

本例的订单约定是名称和数量。数量必须是 1–100 的安全整数，名称去掉首尾空白后非空。这是本例制定的业务规则，不是 JSON 或 TypeScript 自带的限制。

以下片段来自 validation.ts。

```typescript
export interface Order { name: string; quantity: number }
export type Result =
  | { ok: true; value: Order }
  | { ok: false; reason: "json" | "shape" | "business" };
```

Step 1：检查本章正常项目。

```bash
npm run check:22
# 无类型诊断。
```

Step 2：生成当前源码的 JavaScript。

```bash
npm run build:22
# 类型错误时不生成新输出。
```

Step 3：执行本章运行入口。

```bash
npm run run:22
# 输出示例分支，并完成三项运行测试。
```

## 2 结构守卫与业务约束分开

类型守卫先接受非 null 对象或函数，再确认两个属性的运行时类型。value is Order 是类型谓词：true 表示属于该类型，false 表示不属于；编译器不会证明手写谓词满足这个双向关系，因此不能把更严格的业务规则混在里面。

isOrder 只建立结构依据；带 name 和 quantity 的数组或函数也满足 Order 的结构。validOrder 才拒绝数组、函数，并检查范围、安全整数与名称；NaN、Infinity 或零虽是 number，却不符合业务规则。普通 [] 因为缺少必需字段而不通过守卫。附加属性允许存在，这不是密封对象检查。

```typescript
export function isOrder(value: unknown): value is Order {
  return (typeof value === "object" && value !== null || typeof value === "function")
    && "name" in value && typeof value.name === "string"
    && "quantity" in value && typeof value.quantity === "number";
}
export function assertOrder(value: unknown): asserts value is Order {
  if (!isOrder(value)) throw new TypeError("order shape");
}
export function validOrder(value: Order): boolean {
  return typeof value !== "function" && !Array.isArray(value)
    && value.name.trim().length > 0 && Number.isSafeInteger(value.quantity)
    && value.quantity >= 1 && value.quantity <= 100;
}
```

## 3 用可辨识联合表达解析结果

沿输入进入系统的路径检查，才能知道失败来自语法、字段结构，还是业务规则。

Result 用 ok 区分处理结果：成功分支携带 value，失败分支携带具体 reason。图中的三个失败出口分别对应 json、shape、business；它们保留了失败发生的阶段，不用空订单冒充成功。

try 只包住 JSON.parse；catch 中以 instanceof SyntaxError 识别语法失败，其他异常继续抛出，不把未知异常全部吞成“格式不对”。assertOrder 是另一种接口：检查不成立就抛 TypeError，正常返回才收窄调用方。它没有承诺业务校验通过。

![外部 JSON 逐层进入业务数据。每一层回答不同问题；前一层通过，不代表后一层也通过。](image/illustration/22-01-external-data-boundary.svg)

图示说明（依据篇末官方文档自绘）：横向箭头表示前一层成功后继续；向下箭头表示该层失败后返回对应 reason。

把下面四条输入分别沿图追踪，再用 main.ts 的输出核对；不要用 as Order 跳过任一运行时检查。

```typescript
export function parseOrder(text: string): Result {
  let value: unknown;
  try { value = JSON.parse(text); }
  catch (error: unknown) {
    if (error instanceof SyntaxError) return { ok: false, reason: "json" };
    throw error;
  }
  if (!isOrder(value)) return { ok: false, reason: "shape" };
  if (!validOrder(value)) return { ok: false, reason: "business" };
  return { ok: true, value };
}
```

以下片段来自 main.ts。

```typescript
import { parseOrder, assertOrder } from "./validation.js";
// 下面四次分别输出 笔:2、json、shape、business。
for (const text of ['{"name":"笔","quantity":2}', '{', 'null', '{"name":" ","quantity":0}']) {
  const result = parseOrder(text);
  console.log(result.ok ? result.value.name + ":" + result.value.quantity : result.reason);
}
const candidate: unknown = { name: "纸", quantity: 1 };
assertOrder(candidate);
console.log(candidate.name); // 纸：断言函数正常返回后才能访问 name。
try { assertOrder(null); }
catch (error: unknown) {
  // 输入 null 时输出 order shape。
  console.log(error instanceof Error ? error.message : "非 Error 异常");
}
```

## 4 测试结构、范围和失败原因

测试分成三层：结构守卫拒绝缺字段与字段类型错误；业务检查覆盖上下边界、分数和非有限数；解析测试核对三个失败原因。仅测试一个正常对象会漏掉谓词写错时的主要风险。

以下片段来自 validation.test.ts。

```typescript
import test from "node:test";
import assert from "node:assert/strict";
import { isOrder, assertOrder, validOrder, parseOrder } from "./validation.js";
// 先看类型结构：字段名和字段类型满足 Order，不等于符合业务约定。
test("结构边界和断言", () => {
  for (const value of [null, [], 3, {}, { name: "a", quantity: "2" }]) {
    assert.equal(isOrder(value), false);
    assert.throws(() => assertOrder(value), { name: "TypeError", message: "order shape" });
  }
  assert.equal(isOrder({ name: "a", quantity: 1, extra: true }), true);
  // 数组和函数也可带同名属性；这里专门对照结构谓词与业务校验。
  for (const value of [Object.assign([], { name: "a", quantity: 1 }),
    Object.assign(() => {}, { quantity: 1 })]) {
    assert.equal(isOrder(value), true);
    assert.equal(validOrder(value), false);
  }
});
// 再固定结构，只改变数量范围或名字内容，定位业务规则的拒绝原因。
test("业务边界", () => {
  for (const quantity of [0, -1, 1.5, 101, NaN, Infinity]) {
    assert.equal(validOrder({ name: "a", quantity }), false);
  }
  for (const quantity of [1, 100]) assert.equal(validOrder({ name: "a", quantity }), true);
  assert.equal(validOrder({ name: " ", quantity: 1 }), false);
});
// 最后从 JSON 文本进入完整流程：语法、结构、业务依次失败。
test("解析的三个失败层次", () => {
  assert.deepEqual(parseOrder('{'), { ok: false, reason: "json" });
  assert.deepEqual(parseOrder('[]'), { ok: false, reason: "shape" });
  assert.deepEqual(parseOrder('{"name":"a","quantity":0}'), { ok: false, reason: "business" });
  assert.deepEqual(parseOrder('{"name":"a","quantity":1}'), { ok: true, value: { name: "a", quantity: 1 } });
});
```

## 5 DOM 查询不能仅靠类型参数保证节点种类

浏览器 lib 声明使编译器认识 document，不会在 Node.js 中创建 DOM。浏览器项目使用 DOM 与 ES2025 声明，types 为空，和 Node 项目分开。

查询 #quantity 可能返回 null，也可能返回错误标签。instanceof HTMLInputElement 同时检查存在性和节点种类；只写泛型 querySelector 或非空断言不会插入这种校验。

事件的 target 可能来自子节点；currentTarget 是当前处理器所绑定的对象，但 Event 的声明仍允许 EventTarget 或 null。这里同步读取后再次检查节点类型。valueAsNumber 为空或不可转换时可得到 NaN，数量规则因此不能只写大于零。

以下片段来自 browser.ts。

```typescript
const input = document.querySelector("#quantity");
const output = document.querySelector("#result");
if (!(input instanceof HTMLInputElement) || !(output instanceof HTMLOutputElement)) {
  throw new TypeError("页面需要 input#quantity 和 output#result");
}
input.addEventListener("input", (event: Event) => {
  const target = event.currentTarget;
  if (!(target instanceof HTMLInputElement)) return;
  const quantity = target.valueAsNumber;
  // 输入 1 或 100 后分别显示“数量有效：1”“数量有效：100”。
  // 清空，或输入 0、1.5、101 后显示“请输入 1–100 的整数”。
  output.textContent = Number.isSafeInteger(quantity) && quantity >= 1 && quantity <= 100
    ? "数量有效：" + quantity : "请输入 1–100 的整数";
});
```

以下片段来自 tsconfig.browser.json。

```json
{
  "compilerOptions": {
    "target": "ES2025",
    "module": "ESNext",
    "moduleResolution": "bundler",
    "strict": true,
    "lib": [
      "ES2025",
      "DOM"
    ],
    "types": [],
    "rootDir": ".",
    "outDir": ".build-browser",
    "noEmitOnError": true
  },
  "files": [
    "browser.ts"
  ]
}
```

Step 1：构建浏览器专用代码。

```bash
npm run build:22:browser
# 生成 scripts/22-runtime-validation/.build-browser/browser.js。
```

在浏览器打开配套 index.html，依次输入 1、100、0、1.5、101，再清空。前两项应显示数量有效，其余显示整数范围提示。页面无需服务或外网，脚本不导入模块。

以下片段来自 index.html。

```html
<!doctype html>
<html lang="zh-CN"><meta charset="utf-8"><title>订单数量校验</title>
<h1>订单数量校验</h1>
<label>数量 <input id="quantity" type="number" min="1" max="100" step="1"></label>
<output id="result" aria-live="polite">请输入数量</output>
<script src="./.build-browser/browser.js"></script>
</html>
```

## 6 错误分支和异常变量仍需收窄

strict 下的 catch 变量不能假定是 Error，因为 JavaScript 可以抛出任意值。本例显式写 unknown，再通过 instanceof Error 读取 message。同样，Result 没有经过 ok 判断时不能读取 value。

下面文件仅进入错误配置，不作为运行入口。

以下片段来自 type-errors.ts。

```typescript
import { parseOrder } from "./validation.js";
const unknownOrder: unknown = { name: "a" };
unknownOrder.name; // TS18046：先检查 unknown。
const result = parseOrder("null");
console.log(result.value); // TS2339：失败分支没有 value。
try { throw "failed"; } catch (error: unknown) { console.log(error.message); } // TS18046；仅检查，不运行。
```

Step 1：核对未收窄的反例。

```bash
npm run errors:22
# 退出 1，包含 TS18046 与 TS2339。
```

## 本章小结

- JSON 语法、对象结构与业务约束解决不同问题；unknown 保留尚未检查的边界。
- 类型谓词和断言函数需要运行时实现及边界测试支撑。
- DOM 类型声明不保证页面节点存在，事件对象也需要相应收窄。

## 练习

1. 将数量上限从 100 改为 50，同步修改 validOrder、browser.ts、HTML max 属性、提示文本和测试数据；1、50 通过，0、51 和 1.5 拒绝。
2. 在不增加断言的条件下，把解析成功订单格式化为字符串；四个 main 输入仍都正常退出。
3. 把页面 input 改成 div 后刷新，确认初始化按 TypeError 失败；恢复 input，再检查清空和数量边界。

### 提示

1. 数量上限在 Node 业务函数、浏览器脚本和 HTML 中分别出现，需要同步。
2. 用 ok 分支缩窄 Result，保持失败分支返回 reason。
3. 这是有意破坏页面前提的校验机制练习，完成后恢复标签。

### 参考解析

1. 所有上限改为 50；原来的有效值 100 改为 50，无效上界 101 改为 51。结构守卫不包含数量范围，因此无需随业务上限修改。
2. 只在 `result.ok` 成立时读取 value 并格式化；其余分支读取 reason。main 的四条输入仍分别得到成功字符串、json、shape、business。
3. div 不通过 HTMLInputElement 检查，初始化抛 TypeError；恢复 input 后，清空得到 NaN 并进入业务提示分支，1 与上限值有效。

## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TypeScript 官方文档 | [5.5 / Inferred Type Predicates](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-5-5.html#inferred-type-predicates) 的 if-and-only-if 语义及手写谓词边界；[Narrowing](https://www.typescriptlang.org/docs/handbook/2/narrowing.html#using-type-predicates) 的类型谓词、断言函数与可辨识联合；[catch unknown](https://www.typescriptlang.org/tsconfig/useUnknownInCatchVariables.html)、[DOM Manipulation](https://www.typescriptlang.org/docs/handbook/dom-manipulation.html#the-queryselector-and-queryselectorall-methods)：收窄与 DOM 查询返回类型。 |
| ECMA-262 第 16 版 | [JSON.parse](https://tc39.es/ecma262/2025/multipage/structured-data.html#sec-json.parse)、[Number.isSafeInteger](https://tc39.es/ecma262/2025/multipage/numbers-and-dates.html#sec-number.issafeinteger)：JSON 语法及安全整数边界。 |
| WHATWG DOM | [Event 接口](https://dom.spec.whatwg.org/#interface-event)：target、currentTarget 与监听器调用。 |
| WHATWG HTML | [valueAsNumber](https://html.spec.whatwg.org/multipage/input.html#dom-input-valueasnumber)：数值输入和 NaN 边界。 |
| Node.js 24.11.0 | [测试运行器](https://nodejs.org/download/release/v24.11.0/docs/api/test.html)、[assert.throws](https://nodejs.org/download/release/v24.11.0/docs/api/assert.html#assertthrowsfn-error-message)：结构和异常测试。 |
